# Notebook 02 — Indonesian Text Classification from Scratch (BiLSTM)

**Goal:** Train a text classifier **from scratch** on Indonesian news articles (id_liputan6).
Demonstrates understanding of tokenization, embeddings, and sequence modeling without relying on pretrained weights.

**Architecture:**
```
Raw text
  → Custom whitespace tokenizer (vocab built from training data)
  → Embedding layer (random init, learned during training)
  → Bidirectional LSTM (2 layers)
  → Dropout → Linear → Category prediction
```

**Dataset:** id_liputan6 — Indonesian news articles with URL-derived categories

**Runtime:** T4 GPU, ~15–20 min

## 1. Setup

In [ ]:
!pip install -q "datasets<3.0.0" scikit-learn matplotlib seaborn

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
REPO_DIR = '/content/drive/MyDrive/indonesian-audio-intelligence'
if not os.path.exists(f'{REPO_DIR}/src'):
    !git clone https://github.com/KimiDandy/indonesian-audio-intelligence {REPO_DIR}
else:
    print(f'Repo already exists at {REPO_DIR}')

import sys
sys.path.append(REPO_DIR)
os.makedirs(f'{REPO_DIR}/checkpoints', exist_ok=True)
print('Ready.')

In [ ]:
import torch
import torch.nn as nn
import numpy as np
import json
import time
from collections import Counter
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import accuracy_score, f1_score, classification_report

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {device}')

## 2. Load Dataset — id_liputan6

In [ ]:
from datasets import load_dataset

print('Loading id_liputan6...')
ds = load_dataset('id_liputan6', 'complete', trust_remote_code=True)
print(ds)

In [ ]:
# Derive category from URL (e.g. /bisnis/, /teknologi/)
CATEGORIES = {
    'bisnis': 0,
    'teknologi': 1,
    'otomotif': 2,
    'bola': 3,
    'health': 4,
    'lifestyle': 5,
    'showbiz': 6,
    'regional': 7,
}
IDX_TO_LABEL = {v: k for k, v in CATEGORIES.items()}
NUM_CLASSES = len(CATEGORIES)

def extract_label(url: str) -> int:
    for cat, idx in CATEGORIES.items():
        if f'/{cat}/' in url.lower():
            return idx
    return -1

def process_split(split_name, max_samples=10_000):
    texts, labels = [], []
    for item in ds[split_name]:
        lbl = extract_label(item.get('url', ''))
        if lbl == -1:
            continue
        texts.append(item['clean_article'][:512])
        labels.append(lbl)
        if len(texts) >= max_samples:
            break
    return texts, labels

train_texts, train_labels = process_split('train', 8000)
val_texts,   val_labels   = process_split('validation', 1500)
test_texts,  test_labels  = process_split('test', 1500)

print(f'Train: {len(train_texts)}, Val: {len(val_texts)}, Test: {len(test_texts)}')
print('Class distribution:', Counter(train_labels))

## 3. Custom Tokenizer (from scratch)

In [ ]:
class SimpleTokenizer:
    """Whitespace tokenizer with vocabulary built purely from training data."""
    PAD = '<PAD>'
    UNK = '<UNK>'

    def __init__(self, max_vocab: int = 10_000):
        self.max_vocab = max_vocab
        self.word2idx: dict = {}
        self.idx2word: dict = {}

    def build_vocab(self, texts: list[str]) -> None:
        counter: Counter = Counter()
        for text in texts:
            counter.update(text.lower().split())
        vocab = [self.PAD, self.UNK] + [
            w for w, _ in counter.most_common(self.max_vocab - 2)
        ]
        self.word2idx = {w: i for i, w in enumerate(vocab)}
        self.idx2word = {i: w for w, i in self.word2idx.items()}
        print(f'Vocab size: {len(self.word2idx)}')

    def encode(self, text: str, max_len: int = 128) -> list[int]:
        tokens = text.lower().split()[:max_len]
        ids = [self.word2idx.get(t, self.word2idx[self.UNK]) for t in tokens]
        ids += [self.word2idx[self.PAD]] * (max_len - len(ids))
        return ids

    @property
    def vocab_size(self) -> int:
        return len(self.word2idx)

tokenizer = SimpleTokenizer(max_vocab=15_000)
tokenizer.build_vocab(train_texts)

# Quick sanity check
sample_enc = tokenizer.encode('teknologi kecerdasan buatan indonesia', max_len=10)
print('Sample encoding:', sample_enc)

## 4. PyTorch Dataset & DataLoader

In [ ]:
MAX_LEN = 128
BATCH_SIZE = 64

class TextDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len=128):
        self.labels = labels
        self.encodings = [tokenizer.encode(t, max_len=max_len) for t in texts]

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        return (
            torch.tensor(self.encodings[idx], dtype=torch.long),
            torch.tensor(self.labels[idx], dtype=torch.long),
        )

train_ds = TextDataset(train_texts, train_labels, tokenizer, MAX_LEN)
val_ds   = TextDataset(val_texts,   val_labels,   tokenizer, MAX_LEN)
test_ds  = TextDataset(test_texts,  test_labels,  tokenizer, MAX_LEN)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,  num_workers=2)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False, num_workers=2)
test_loader  = DataLoader(test_ds,  batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

print(f'Batches — train: {len(train_loader)}, val: {len(val_loader)}, test: {len(test_loader)}')

## 5. BiLSTM Model (defined from scratch)

In [ ]:
class LSTMClassifier(nn.Module):
    def __init__(self, vocab_size, num_classes, embed_dim=128, hidden_dim=256,
                 num_layers=2, dropout=0.3, padding_idx=0):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=padding_idx)
        self.lstm = nn.LSTM(
            embed_dim, hidden_dim, num_layers=num_layers,
            batch_first=True, dropout=dropout, bidirectional=True
        )
        self.dropout = nn.Dropout(0.5)
        self.fc = nn.Linear(hidden_dim * 2, num_classes)

    def forward(self, x):
        embedded = self.dropout(self.embedding(x))
        _, (hidden, _) = self.lstm(embedded)
        hidden = torch.cat([hidden[-2], hidden[-1]], dim=1)  # concat fwd + bwd
        return self.fc(self.dropout(hidden))

model = LSTMClassifier(
    vocab_size=tokenizer.vocab_size,
    num_classes=NUM_CLASSES,
).to(device)

total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Total params: {total_params:,}')
print(f'Trainable:    {trainable_params:,}')

## 6. Training Loop

In [ ]:
EPOCHS = 10
LR = 1e-3

optimizer = torch.optim.Adam(model.parameters(), lr=LR)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, patience=2, factor=0.5)
criterion = nn.CrossEntropyLoss()

history = {'train_loss': [], 'val_loss': [], 'train_acc': [], 'val_acc': []}

best_val_f1 = 0.0
best_state = None

def evaluate(loader):
    model.eval()
    all_preds, all_labels, total_loss = [], [], 0.0
    with torch.no_grad():
        for x, y in loader:
            x, y = x.to(device), y.to(device)
            logits = model(x)
            total_loss += criterion(logits, y).item()
            all_preds.extend(logits.argmax(dim=1).cpu().tolist())
            all_labels.extend(y.cpu().tolist())
    acc = accuracy_score(all_labels, all_preds)
    f1  = f1_score(all_labels, all_preds, average='macro', zero_division=0)
    return total_loss / len(loader), acc, f1, all_preds, all_labels

start_time = time.time()

for epoch in range(1, EPOCHS + 1):
    model.train()
    train_loss, correct, total = 0.0, 0, 0
    for x, y in train_loader:
        x, y = x.to(device), y.to(device)
        optimizer.zero_grad()
        logits = model(x)
        loss = criterion(logits, y)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        train_loss += loss.item()
        correct += (logits.argmax(1) == y).sum().item()
        total += y.size(0)

    train_acc = correct / total
    val_loss, val_acc, val_f1, _, _ = evaluate(val_loader)
    scheduler.step(val_loss)

    history['train_loss'].append(train_loss / len(train_loader))
    history['val_loss'].append(val_loss)
    history['train_acc'].append(train_acc)
    history['val_acc'].append(val_acc)

    if val_f1 > best_val_f1:
        best_val_f1 = val_f1
        best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}

    print(f'Epoch {epoch:2d}/{EPOCHS} | '
          f'Train Loss: {train_loss/len(train_loader):.4f} Acc: {train_acc:.3f} | '
          f'Val Loss: {val_loss:.4f} Acc: {val_acc:.3f} F1: {val_f1:.3f}')

elapsed = (time.time() - start_time) / 60
print(f'\nTraining complete in {elapsed:.1f} min. Best Val F1: {best_val_f1:.4f}')

## 7. Evaluate on Test Set

In [ ]:
# Load best checkpoint
model.load_state_dict({k: v.to(device) for k, v in best_state.items()})

_, test_acc, test_f1, test_preds, test_true = evaluate(test_loader)

print('=' * 50)
print(f'  LSTM from Scratch — Test Results')
print(f'  Accuracy:  {test_acc:.4f} ({test_acc*100:.1f}%)')
print(f'  F1 Macro:  {test_f1:.4f}')
print('=' * 50)

print()
print(classification_report(
    test_true, test_preds,
    target_names=list(CATEGORIES.keys()),
    zero_division=0
))

In [ ]:
from src.utils.visualize import plot_training_curves, plot_confusion_matrix

fig1 = plot_training_curves(
    history['train_loss'], history['val_loss'],
    history['train_acc'], history['val_acc'],
    title='BiLSTM from Scratch'
)
fig1.savefig(f'{REPO_DIR}/lstm_training_curves.png', dpi=150, bbox_inches='tight')

fig2 = plot_confusion_matrix(
    test_true, test_preds,
    label_names=list(CATEGORIES.keys()),
    title='BiLSTM Confusion Matrix'
)
fig2.savefig(f'{REPO_DIR}/lstm_confusion_matrix.png', dpi=150, bbox_inches='tight')

In [ ]:
# Save checkpoint and results for notebook 03/04
torch.save({
    'model_state_dict': best_state,
    'tokenizer_word2idx': tokenizer.word2idx,
    'vocab_size': tokenizer.vocab_size,
    'num_classes': NUM_CLASSES,
    'categories': CATEGORIES,
    'test_accuracy': test_acc,
    'test_f1_macro': test_f1,
    'train_time_min': elapsed,
}, f'{REPO_DIR}/checkpoints/lstm_best.pt')

print('Checkpoint saved.')
print(f'Results to compare in Notebook 03:')
print(f'  LSTM Accuracy: {test_acc:.4f}')
print(f'  LSTM F1 Macro: {test_f1:.4f}')